## Anna Giczewska ST544 - Project 2
Date 3/27/2026  

This project has two main parts. In Part I, I create and test a custom PySpark class called SparkDataCheck that helps check and summarize data quality in Spark SQL style DataFrames. In Part II, I analyze NFL quarterback data using both pandas-on-Spark and Spark SQL DataFrames.

Throughout the notebook, I include code, output, and short explanations to describe what I am doing and why each step is included.

## Part I - Creating and Testing the SparkDataCheck Class

In this part of the project, I work with a Spark SQL style DataFrame and create a custom Python class that wraps the DataFrame and provides methods for checking data quality and summarizing the data.

The class is stored in a separate .py file, and I import it into this notebook so I can test each method here.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#start Spark Session for this project
spark = SparkSession.builder.appName("Project2").getOrCreate()
#turn off ANSI mode to avoid certainstrict Spark errors
spark.conf.set("spark.sql.ansi.enabled", "false")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 22:11:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#import importlib so I can reload my Python file after edditing it
import importlib
#import my claddfile
import spark_data_check
#reload the file so the notebook uses the newest version
importlib.reload(spark_data_check)

#Import the SparkDataCheck class from my file
from spark_data_check import SparkDataCheck

# Show which file is being imported
print(spark_data_check.__file__)

# Check which methods exist on the class itself
print(hasattr(SparkDataCheck, "from_csv"))
print(hasattr(SparkDataCheck, "from_pandas"))
print(hasattr(SparkDataCheck, "check_numeric_range"))
print(hasattr(SparkDataCheck, "check_string_levels"))
print(hasattr(SparkDataCheck, "check_missing"))
print(hasattr(SparkDataCheck, "summarize_numeric_min_max"))
print(hasattr(SparkDataCheck, "count_string_levels"))

/home/jupyter-agiczew@ncsu.edu/st554_project2/spark_data_check.py
True
True
True
True
True
True
True


In [3]:
air_url = "https://www4.stat.ncsu.edu/online/datasets/air.csv"
#read the CSV from the website into a pandas DataFrame
air_download = pd.read_csv(air_url)
#save a local copy of air.csv file in my project folder
air_download.to_csv("air.csv", index=False)

# Check the first few rows and column names
print(air_download.columns.tolist())
air_download.head()

['Unnamed: 0', 'Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [4]:
#createa SparkData Object by reading CSV with Spark
air_check = SparkDataCheck.from_csv(spark, "air.csv")
#confirm object creation
type(air_check)

spark_data_check.SparkDataCheck

In [5]:
#show data
air_check.df.show(5)
air_check.df.printSchema()

+----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|2026-03-27 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|2026-03-27 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|2026-03-27 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         

In [6]:
#read CSV into pandas DataFrame
air_pd = pd.read_csv("air.csv")
#Create a SparkDataCheck object from the pandas DataFrame
air_check_pd = SparkDataCheck.from_pandas(spark, air_pd)
#confirm object was created
type(air_check_pd)


spark_data_check.SparkDataCheck

In [7]:
air_check_pd.df.show(5)

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

In [8]:
# Create helper columns for testing the validation methods
air_check.df = (
    air_check.df
    .withColumn(
        "time_period",
        F.when(F.col("Time") < "12:00:00", "morning")
         .when(F.col("Time") < "18:00:00", "afternoon")
         .otherwise("evening")
    )
    .withColumn(
        "temp_group",
        F.when(F.col("T").isNull(), F.lit(None))
         .when(F.col("T") < 10, "cold")
         .when(F.col("T") < 20, "mild")
         .otherwise("warm")
    )
    .withColumn(
        "co_clean",
        F.when(F.col("CO(GT)") == -200, F.lit(None))
         .otherwise(F.col("CO(GT)"))
    )
)

# Show the new helper columns
air_check.df.select("Time", "T", "CO(GT)", "time_period", "temp_group", "co_clean").show(10)

+-------------------+----+------+-----------+----------+--------+
|               Time|   T|CO(GT)|time_period|temp_group|co_clean|
+-------------------+----+------+-----------+----------+--------+
|2026-03-27 18:00:00|13.6|   2.6|    evening|      mild|     2.6|
|2026-03-27 19:00:00|13.3|   2.0|    evening|      mild|     2.0|
|2026-03-27 20:00:00|11.9|   2.2|    evening|      mild|     2.2|
|2026-03-27 21:00:00|11.0|   2.2|    evening|      mild|     2.2|
|2026-03-27 22:00:00|11.2|   1.6|    evening|      mild|     1.6|
|2026-03-27 23:00:00|11.2|   1.2|    evening|      mild|     1.2|
|2026-03-27 00:00:00|11.3|   1.2|    morning|      mild|     1.2|
|2026-03-27 01:00:00|10.7|   1.0|    morning|      mild|     1.0|
|2026-03-27 02:00:00|10.7|   0.9|    morning|      mild|     0.9|
|2026-03-27 03:00:00|10.3|   0.6|    morning|      mild|     0.6|
+-------------------+----+------+-----------+----------+--------+
only showing top 10 rows


**Method 1: Check check_numeric_range()**

In [24]:
# Check whether temperature is in a reasonable range
air_check.check_numeric_range("T", lower=0, upper=40, new_col_name="temp_ok")
air_check.df.select("T", "temp_ok").show(10)

+----+-------+
|   T|temp_ok|
+----+-------+
|13.6|   true|
|13.3|   true|
|11.9|   true|
|11.0|   true|
|11.2|   true|
|11.2|   true|
|11.3|   true|
|10.7|   true|
|10.7|   true|
|10.3|   true|
+----+-------+
only showing top 10 rows


In [25]:
# Check whether relative humidity is at most 100
air_check.check_numeric_range("RH", upper=100, new_col_name="rh_ok")
air_check.df.select("RH", "rh_ok").show(10)

+----+-----+
|  RH|rh_ok|
+----+-----+
|48.9| true|
|47.7| true|
|54.0| true|
|60.0| true|
|59.6| true|
|59.2| true|
|56.8| true|
|60.0| true|
|59.7| true|
|60.2| true|
+----+-----+
only showing top 10 rows


In [26]:
# Test a bad input using a string column
air_check.check_numeric_range("time_period", lower=1, upper=10)

Column 'time_period' is not numeric.


In [27]:
# Check whether AH is at least 0.5
air_check.check_numeric_range("AH", lower=0.5, new_col_name="AH_ge_0_5")
air_check.df.select("AH", "AH_ge_0_5").show(10)

+------+---------+
|    AH|AH_ge_0_5|
+------+---------+
|0.7578|     true|
|0.7255|     true|
|0.7502|     true|
|0.7867|     true|
|0.7888|     true|
|0.7848|     true|
|0.7603|     true|
|0.7702|     true|
|0.7648|     true|
|0.7517|     true|
+------+---------+
only showing top 10 rows


In [28]:
# Check whether co_clean is between 0 and 5
air_check.check_numeric_range("co_clean", lower=0, upper=5, new_col_name="co_clean_0_5")
air_check.df.select("co_clean", "co_clean_0_5").show(10)

+--------+------------+
|co_clean|co_clean_0_5|
+--------+------------+
|     2.6|        true|
|     2.0|        true|
|     2.2|        true|
|     2.2|        true|
|     1.6|        true|
|     1.2|        true|
|     1.2|        true|
|     1.0|        true|
|     0.9|        true|
|     0.6|        true|
+--------+------------+
only showing top 10 rows


**Method 2: Check check_string_levels()**

In [29]:
# Check whether time_period belongs to the expected levels
air_check.check_string_levels(
    "time_period",
    ["morning", "afternoon", "evening"],
    new_col_name="time_period_ok"
)
air_check.df.select("time_period", "time_period_ok").show(10)

+-----------+--------------+
|time_period|time_period_ok|
+-----------+--------------+
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    morning|          true|
|    morning|          true|
|    morning|          true|
|    morning|          true|
+-----------+--------------+
only showing top 10 rows


In [30]:
# Check whether temp_group belongs to the expected levels
air_check.check_string_levels(
    "temp_group",
    ["cold", "mild", "warm"],
    new_col_name="temp_group_ok"
)
air_check.df.select("temp_group", "temp_group_ok").show(10)

+----------+-------------+
|temp_group|temp_group_ok|
+----------+-------------+
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
+----------+-------------+
only showing top 10 rows


In [31]:
# Test a bad input using a numeric column
air_check.check_string_levels("T", ["cold", "warm"])

Column 'T' is not a string column.


In [32]:
# Check stricter valid levels for time_period
air_check.check_string_levels(
    "time_period",
    ["morning", "afternoon"],
    new_col_name="time_period_day_only"
)
air_check.df.select("time_period", "time_period_day_only").show(10)

+-----------+--------------------+
|time_period|time_period_day_only|
+-----------+--------------------+
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    morning|                true|
|    morning|                true|
|    morning|                true|
|    morning|                true|
+-----------+--------------------+
only showing top 10 rows


In [33]:
# Check stricter valid levels for temp_group
air_check.check_string_levels(
    "temp_group",
    ["cold", "warm"],
    new_col_name="temp_group_cold_warm"
)
air_check.df.select("temp_group", "temp_group_cold_warm").show(10)

+----------+--------------------+
|temp_group|temp_group_cold_warm|
+----------+--------------------+
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
+----------+--------------------+
only showing top 10 rows


**Method 3: check_missing()**

In [34]:
# Check missing values in the cleaned CO column
air_check.check_missing("co_clean", new_col_name="co_missing")
air_check.df.select("CO(GT)", "co_clean", "co_missing").show(15)

+------+--------+----------+
|CO(GT)|co_clean|co_missing|
+------+--------+----------+
|   2.6|     2.6|     false|
|   2.0|     2.0|     false|
|   2.2|     2.2|     false|
|   2.2|     2.2|     false|
|   1.6|     1.6|     false|
|   1.2|     1.2|     false|
|   1.2|     1.2|     false|
|   1.0|     1.0|     false|
|   0.9|     0.9|     false|
|   0.6|     0.6|     false|
|-200.0|    NULL|      true|
|   0.7|     0.7|     false|
|   0.7|     0.7|     false|
|   1.1|     1.1|     false|
|   2.0|     2.0|     false|
+------+--------+----------+
only showing top 15 rows


In [35]:
# Create another column with some artificial missing values for testing
air_check.df = air_check.df.withColumn(
    "ah_test",
    F.when(F.col("AH") < 0.5, F.lit(None)).otherwise(F.col("AH"))
)

# Check missing values in the new test column
air_check.check_missing("ah_test", new_col_name="ah_test_missing")
air_check.df.select("AH", "ah_test", "ah_test_missing").show(15)

+------+-------+---------------+
|    AH|ah_test|ah_test_missing|
+------+-------+---------------+
|0.7578| 0.7578|          false|
|0.7255| 0.7255|          false|
|0.7502| 0.7502|          false|
|0.7867| 0.7867|          false|
|0.7888| 0.7888|          false|
|0.7848| 0.7848|          false|
|0.7603| 0.7603|          false|
|0.7702| 0.7702|          false|
|0.7648| 0.7648|          false|
|0.7517| 0.7517|          false|
|0.7465| 0.7465|          false|
|0.7366| 0.7366|          false|
|0.7353| 0.7353|          false|
|0.7417| 0.7417|          false|
|0.7408| 0.7408|          false|
+------+-------+---------------+
only showing top 15 rows


In [36]:
# Check missingness in T
air_check.check_missing("T", new_col_name="T_missing")
air_check.df.select("T", "T_missing").show(10)

+----+---------+
|   T|T_missing|
+----+---------+
|13.6|    false|
|13.3|    false|
|11.9|    false|
|11.0|    false|
|11.2|    false|
|11.2|    false|
|11.3|    false|
|10.7|    false|
|10.7|    false|
|10.3|    false|
+----+---------+
only showing top 10 rows


In [37]:
# Check missingness in time_period
air_check.check_missing("time_period", new_col_name="time_period_missing")
air_check.df.select("time_period", "time_period_missing").show(10)

+-----------+-------------------+
|time_period|time_period_missing|
+-----------+-------------------+
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    morning|              false|
|    morning|              false|
|    morning|              false|
|    morning|              false|
+-----------+-------------------+
only showing top 10 rows


In [38]:
# Check missingness in temp_group
air_check.check_missing("temp_group", new_col_name="temp_group_missing")
air_check.df.select("temp_group", "temp_group_missing").show(10)

+----------+------------------+
|temp_group|temp_group_missing|
+----------+------------------+
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
+----------+------------------+
only showing top 10 rows


**Method 4: Check summarize_numeric_min_max()**

In [12]:
# Summarize one numeric column without grouping
result1 = air_check.summarize_numeric_min_max("T")

# Check the output type
print(type(result1))

# Show the result
result1

<class 'pandas.core.frame.DataFrame'>


,T_min,T_max
0,-200.0,44.6


In [13]:
# Summarize one numeric column grouped by a string column
result2 = air_check.summarize_numeric_min_max("RH", group_by="time_period")

# Check the output type
print(type(result2))

# Show the result
result2

<class 'pandas.core.frame.DataFrame'>


,time_period,RH_min,RH_max
0,afternoon,-200.0,88.7
1,evening,-200.0,87.2
2,morning,-200.0,87.1


In [14]:
# Summarize all numeric columns without grouping
result3 = air_check.summarize_numeric_min_max()

# Check the output type
print(type(result3))

# Show the result
result3

26/03/27 22:11:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


<class 'pandas.core.frame.DataFrame'>


,Unnamed: 0_min,Unnamed: 0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,T_min,T_max,RH_min,RH_max,AH_min,AH_max,co_clean_min,co_clean_max,ah_test_min,ah_test_max
0,0,9356,-200.0,11.9,-200,2040,-200,1189,-200.0,63.7,...,-200.0,44.6,-200.0,88.7,-200.0,2.231,0.1,11.9,0.5002,2.231


In [15]:
# Summarize all numeric columns grouped by a string column
result4 = air_check.summarize_numeric_min_max(group_by="time_period")

# Check the output type
print(type(result4))

# Show the result
result4

<class 'pandas.core.frame.DataFrame'>


,time_period,Unnamed: 0_min,Unnamed: 0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,...,T_min,T_max,RH_min,RH_max,AH_min,AH_max,co_clean_min,co_clean_max,ah_test_min,ah_test_max
0,afternoon,18,9356,-200.0,8.4,-200,1915,-200,832,-200.0,...,-200.0,44.6,-200.0,88.7,-200.0,2.2310,0.1,8.4,0.5002,2.2310
1,evening,0,9341,-200.0,11.9,-200,2040,-200,1189,-200.0,...,-200.0,40.6,-200.0,87.2,-200.0,2.1107,0.1,11.9,0.5007,2.1107
2,morning,6,9353,-200.0,8.1,-200,1961,-200,1129,-200.0,...,-200.0,38.3,-200.0,87.1,-200.0,2.1719,0.1,8.1,0.5007,2.1719


In [16]:
# Test a bad input using a string column
air_check.summarize_numeric_min_max("time_period")

Column 'time_period' is not numeric.


**Method 5: Check count_string_levels()**

In [17]:
# Count levels of one string column
result5 = air_check.count_string_levels("time_period")

# Check the output type
print(type(result5))

# Show the result
result5

<class 'pandas.core.frame.DataFrame'>


,time_period,count
0,afternoon,2337
1,evening,2340
2,morning,4680


In [18]:
# Count levels of another string column
result6 = air_check.count_string_levels("temp_group")

#show the results
result6

,temp_group,count
0,cold,2032
1,mild,3577
2,warm,3748


In [19]:
# Count combinations of two string columns
result7 = air_check.count_string_levels("time_period", "temp_group")

# Check the output type
print(type(result7))

# Show the result
result7

<class 'pandas.core.frame.DataFrame'>


,time_period,temp_group,count
0,afternoon,cold,281
1,afternoon,mild,761
2,afternoon,warm,1295
3,evening,cold,484
4,evening,mild,884
5,evening,warm,972
6,morning,cold,1267
7,morning,mild,1932
8,morning,warm,1481


In [20]:
# Test a bad input using a numeric column
air_check.count_string_levels("T")

Column 'T' is numeric.


In [21]:
# Test a bad input where the second column is numeric
air_check.count_string_levels("time_period", "RH")

Column 'RH' is numeric.


**Example using object created from pandas**

In [39]:
# One example using the object created from a pandas DataFrame
result_pd = air_check_pd.summarize_numeric_min_max("T")

# Check the output type
print(type(result_pd))

# Show the result
result_pd

<class 'pandas.core.frame.DataFrame'>


,T_min,T_max
0,-200.0,44.6
